## Duration

Modified duration is the sensitivity of bond price on interest rate change.

bond PV = Coupon PV + Principal PV = $c \sum_{i=1}^{N} e^{-\frac{i}{m} r_i} + 100 e^{-\frac{N}{m} r_N}$

Where $r_n$ is zero rate

TODO: derive duration

## Treasury zero rate estimation using t-bond data


### t-bond data

In [3]:
from x01_data import parse_macroevent, bond_pricing

country = "KR"
df = parse_macroevent(f"x01_gvt_{country}.csv")

In [ ]:
## Bond Pricing. It uses B-bond auction price as market price of bond.

from dateutil import relativedelta
import pandas as pd 

df["Bond principal"] = 100
df["Maturity Date"] = [pd.to_datetime(x) for x in ["2027-09","2028-12","2030-09","2035-12","2045-09","2055-09","2074-09",]]
df["Issue Date"] = [(y - pd.offsets.DateOffset(years=x)) for x,y in df["Maturity Date"].items()]
df["Annual coupon"] = [x*2 for x in [2.250,2.750,2.500,3.250,2.750,2.625,2.750]]
df["TTM"] = df.apply(lambda x: relativedelta.relativedelta(x["Maturity Date"], x["Settle Date"]), axis=1).map(lambda x: x.years + x.months/12 + x.days / 360)
# Let's not forget to multiply 0.01
df["Bond price (Inferred)"] = df.apply(lambda x: bond_pricing(100, 2, 0.01 * x["Annual coupon"], 0.01 * x["Bond Yield"], x["Issue Date"], x["Maturity Date"], x["Settle Date"])[1], axis=1)

df

,Settle Date,Bond Yield,Bond principal,Maturity Date,Issue Date,Annual coupon,TTM,Bond price (Inferred)
Maturity,,,,,,,,
2,2025-12-01,2.876,100,2027-09-01,2025-09-01,4.50,1.750000,103.874850
3,2025-12-08,2.970,100,2028-12-01,2025-12-01,5.50,2.980556,107.270996
5,2025-12-22,3.245,100,2030-09-01,2025-09-01,5.00,4.694444,109.118230
10,2025-12-15,3.410,100,2035-12-01,2025-12-01,6.50,9.961111,126.159855
20,2025-12-23,3.365,100,2045-09-01,2025-09-01,5.50,19.691667,132.262560
30,2025-12-02,3.225,100,2055-09-01,2025-09-01,5.25,29.750000,139.871567
50,2025-12-12,3.140,100,2074-09-01,2024-09-01,5.50,48.722222,NaN


In [8]:
# Similar with Hull Table 4.3
df1 = df[["Bond principal", "TTM", "Annual coupon", "Bond price (Inferred)", "Bond Yield"]]
df1

,Bond principal,TTM,Annual coupon,Bond price (Inferred),Bond Yield
Maturity,,,,,
2,100,1.750000,4.50,103.874850,2.876
3,100,2.980556,5.50,107.270996,2.970
5,100,4.694444,5.00,109.118230,3.245
10,100,9.961111,6.50,126.159855,3.410
20,100,19.691667,5.50,132.262560,3.365
30,100,29.750000,5.25,139.871567,3.225
50,100,48.722222,5.50,NaN,3.140


### Bootstrap method 

In this example, some data is interpolated. 

The result can be improved by using more data, or better model 

In [ ]:
# Let's use linear interopolation model to infer the zero rate curve from the data points.
# from scipy.interpolate import interp1d
class Interp1d:
    def __init__(self):
        self.xs:List[float] = []
        self.ys:List[float] = [] 
    def interpolate(self, x:float) -> float:
        return numpy.interp(x, self.xs, self.ys)
    def add_data(self, x:float, y:float):
        self.xs += [x]
        self.ys += [y]

def dates_into_year(d1, d2):
    assert(d1 < d2)
    x = relativedelta.relativedelta(d2, d1)
    return x.years + x.months/12 + x.days / 360

def get_coupon_dates(issue_date, maturity_date, freq):
    months = {1:12, 2:6, 4:3, 12:1}[freq]
    return pd.date_range(issue_date, maturity_date, freq=pd.DateOffset(months=months))[1:]

d0 = pd.to_datetime("2025-12-01") # starting date we set for the zero rate 

for i,x in df.iterrows(): # Bootstrap
    print(x)
    principal_amt = 100
    freq = 2 

    coupon_dates = get_coupon_dates(x["Issue Date"], x["Maturity Date"], freq=freq)
    coupon_amt = principal_amt * x["Annual coupon"] / freq

    coupon_dates_in_years = [dates_into_year(x["Settle Date"], y) for y in coupon_dates]
    print(coupon_dates)
    print(x["Settle Date"])
    print(coupon_dates_in_years)
    
    # x["Bond Yield"] is not continuous compounding version (it is based on acution notice document)
    # x["Bond Price"] = coupon_amt * np.sum(np.exp ( continuous_bond_yield -  * x["coupon_dates_in_years"] )) + principal_amt * np.exp ( continuous_bond_yield * x["coupon_dates_in_years"].iloc[-1] )


    # exp(-1/m y) + exp(-2/m y) + exp(-2/m y) ...  

    # x["TTM"]
    # x["Annual coupon"]

Settle Date              2025-12-01 00:00:00
Bond Yield                             2.876
Bond principal                           100
Maturity Date            2027-09-01 00:00:00
Issue Date               2025-09-01 00:00:00
Annual coupon                            4.5
TTM                                     1.75
Bond price (Inferred)              103.87485
Name: 2, dtype: object
DatetimeIndex(['2026-03-01', '2026-09-01', '2027-03-01', '2027-09-01'], dtype='datetime64[ns]', freq='<DateOffset: months=6>')
2025-12-01 00:00:00
[0.25, 0.75, 1.25, 1.75]
Settle Date              2025-12-08 00:00:00
Bond Yield                              2.97
Bond principal                           100
Maturity Date            2028-12-01 00:00:00
Issue Date               2025-12-01 00:00:00
Annual coupon                            5.5
TTM                                 2.980556
Bond price (Inferred)             107.270996
Name: 3, dtype: object
DatetimeIndex(['2026-06-01', '2026-12-01', '2027-06-01', '2027

AssertionError: 